# Supplementary Figure S2 — competing programs and PBRM1 sensitivity

Competing-program gene sets are declared explicitly. PBRM1 is obtained from
the public CPTAC somatic-mutation export in the paper source manifest.
Within-stratum estimates are emitted only when both strata are estimable; no
P-value threshold is used to decide whether to show them.


In [13]:
from pathlib import Path
import gzip
import hashlib
import json
import re
import tarfile
import warnings

import numpy as np
import pandas as pd
import pyreadr
import statsmodels.api as sm
from scipy.stats import spearmanr

PROJECT_ROOT = Path("/scratch/ar58064/mitoxy")
RAW_MANIFEST_PATH = (
    PROJECT_ROOT / "data" / "provenance" / "paper_public_source_manifest.csv"
)
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "figure_1_clean"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENES = ["BAK1", "BAX", "BID", "SLC7A11"]
CORE3 = ["BAK1", "BAX", "BID"]
RANDOM_SEED = 20260913

raw_manifest = pd.read_csv(RAW_MANIFEST_PATH)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def source_path(dataset, role):
    matches = raw_manifest.loc[
        raw_manifest["dataset"].eq(dataset)
        & raw_manifest["role"].eq(role)
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one public source for ({dataset!r}, {role!r}); "
            f"found {len(matches)}"
        )
    row = matches.iloc[0]
    path = PROJECT_ROOT / str(row["relative_path"])
    if not path.exists():
        raise FileNotFoundError(path)
    if path.stat().st_size != int(row["size_bytes"]):
        raise RuntimeError(f"Source-size mismatch: {path}")
    if sha256_file(path) != str(row["sha256"]):
        raise RuntimeError(f"Source SHA-256 mismatch: {path}")
    return path


RAW = {
    "tcga_xena_expression": source_path(
        "TCGA-KIRC", "tumor RNA expression"
    ),
    "tcga_gdc_clinical_json": source_path(
        "TCGA-KIRC", "clinical API response"
    ),
    "cptac_rna": source_path(
        "CPTAC ccRCC", "tumor RNA expression"
    ),
    "cptac_protein": source_path(
        "CPTAC ccRCC", "tumor protein abundance"
    ),
    "cptac_clinical_xlsx": source_path(
        "CPTAC ccRCC", "clinical supplement"
    ),
    "emtab_expression": source_path(
        "E-MTAB-1980", "ccRCC_exp_log_quantile_normalized.txt"
    ),
    "emtab_clinical_xlsx": source_path(
        "E-MTAB-1980",
        "Sato_2013_Supplementary_Table_1_patient_characteristics.xlsx",
    ),
    "tracerx_tpm": source_path(
        "TRACERx Renal", "regional tumor TPM"
    ),
    "tracerx_annotation": source_path(
        "TRACERx Renal", "regional annotation"
    ),
    "tracerx_clinical": source_path(
        "TRACERx Renal", "clinical annotation"
    ),
}

print("Paper-wide public-source manifest loaded:", RAW_MANIFEST_PATH)
display(pd.DataFrame({"analysis_key": RAW.keys(), "path": RAW.values()}))


Paper-wide public-source manifest loaded: /scratch/ar58064/mitoxy/data/provenance/paper_public_source_manifest.csv


,analysis_key,path
0,tcga_xena_expression,/scratch/ar58064/mitoxy/data/raw/tcga_kirc_pub...
1,tcga_gdc_clinical_json,/scratch/ar58064/mitoxy/data/raw/tcga_kirc_pub...
2,cptac_rna,/scratch/ar58064/mitoxy/data/raw/cptac_ccrcc_p...
3,cptac_protein,/scratch/ar58064/mitoxy/data/raw/cptac_ccrcc_p...
4,cptac_clinical_xlsx,/scratch/ar58064/mitoxy/data/raw/cptac_ccrcc_p...
5,emtab_expression,/scratch/ar58064/mitoxy/data/raw/e_mtab_1980_p...
6,emtab_clinical_xlsx,/scratch/ar58064/mitoxy/data/raw/e_mtab_1980_p...
7,tracerx_tpm,/scratch/ar58064/mitoxy/data/raw/tracerx_renal...
8,tracerx_annotation,/scratch/ar58064/mitoxy/data/raw/tracerx_renal...
9,tracerx_clinical,/scratch/ar58064/mitoxy/data/raw/tracerx_renal...


In [14]:
def zscore(series):
    values = pd.to_numeric(series, errors="coerce")
    return (values - values.mean()) / values.std(ddof=0)


def equal4_score(frame):
    standardized = frame[GENES].apply(zscore, axis=0)
    return standardized.mean(axis=1)


def parse_grade(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"(?:G(?:RADE)?\s*)?([1-4])", str(value), flags=re.I)
    return float(match.group(1)) if match else np.nan


def parse_stage(value):
    if pd.isna(value):
        return np.nan
    text = str(value).upper().replace("STAGE", " ")
    match = re.search(r"\b(IV|III|II|I)\b", text)
    if match:
        return {"I": 1.0, "II": 2.0, "III": 3.0, "IV": 4.0}[match.group(1)]
    match = re.search(r"\b([1-4])\b", text)
    return float(match.group(1)) if match else np.nan


def parse_t_category(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"T([1-4])", str(value).upper())
    return float(match.group(1)) if match else np.nan


def parse_m1(value):
    if pd.isna(value):
        return np.nan
    text = str(value).upper()
    if re.search(r"M1", text):
        return 1.0
    if re.search(r"M0", text):
        return 0.0
    return np.nan


def correlation_row(cohort, endpoint, data, outcome_column):
    complete = data[["mitoxy", outcome_column]].dropna()
    rho, p_value = spearmanr(complete["mitoxy"], complete[outcome_column])
    z_value = np.arctanh(np.clip(rho, -0.999999, 0.999999))
    se = 1 / np.sqrt(len(complete) - 3)
    return {
        "cohort": cohort,
        "endpoint": endpoint,
        "n": len(complete),
        "rho": float(rho),
        "ci_low": float(np.tanh(z_value - 1.95996398454 * se)),
        "ci_high": float(np.tanh(z_value + 1.95996398454 * se)),
        "p_value": float(p_value),
    }


def read_linkedomics(path):
    matrix = pd.read_csv(path, sep="\t", index_col=0, low_memory=False)
    matrix = matrix.apply(pd.to_numeric, errors="coerce")
    matrix.index = [str(value).split("|")[0].strip().upper() for value in matrix.index]
    matrix.columns = [str(value).strip().upper() for value in matrix.columns]
    if matrix.index.duplicated().any():
        matrix = matrix.groupby(level=0, sort=False).mean()
    return matrix


def find_excel_header(path, required_terms, maximum_rows=20):
    preview = pd.read_excel(path, header=None, nrows=maximum_rows)
    best_row, best_score = None, -1
    for row_number, row in preview.iterrows():
        text = " | ".join(row.dropna().astype(str)).lower()
        score = sum(term.lower() in text for term in required_terms)
        if score > best_score:
            best_row, best_score = row_number, score
    return int(best_row)


def column_containing(columns, terms, required=True):
    for column in columns:
        normalized = str(column).strip().lower()
        if any(term.lower() in normalized for term in terms):
            return column
    if required:
        raise KeyError(f"No column matched {terms}; available: {list(columns)}")
    return None


In [15]:
tcga_expression = pd.read_csv(
    RAW["tcga_xena_expression"], sep="\t", index_col=0
)
tcga_expression.index = tcga_expression.index.astype(str).str.upper()

missing = sorted(set(GENES) - set(tcga_expression.index))
assert not missing, f"TCGA expression missing {missing}"

tcga_gene = tcga_expression.loc[GENES].T.apply(pd.to_numeric, errors="coerce")
tcga_gene.index = tcga_gene.index.astype(str).str.upper()

# Primary-tumor samples only. Xena barcodes commonly end in ``-01``;
# some releases append an aliquot letter such as ``-01A``.
primary_mask = tcga_gene.index.str.match(
    r"^TCGA-[A-Z0-9]+-[A-Z0-9]+-01",
    na=False,
)
tcga_gene = tcga_gene.loc[primary_mask].copy()
tcga_gene["sample_id"] = tcga_gene.index.str.slice(0, 12)
tcga_gene = tcga_gene.groupby("sample_id")[GENES].mean()
assert len(tcga_gene) == 533, (
    f"Expected 533 TCGA primary-tumor cases; found {len(tcga_gene)}"
)
tcga_gene["mitoxy"] = equal4_score(tcga_gene)

clinical_object = json.loads(RAW["tcga_gdc_clinical_json"].read_text())
if isinstance(clinical_object, list):
    cases = clinical_object
elif "data" in clinical_object:
    cases = clinical_object["data"].get("hits", clinical_object["data"])
else:
    cases = clinical_object.get("hits", clinical_object)

clinical_rows = []
for case in cases:
    if not isinstance(case, dict):
        continue
    case_id = case.get("submitter_id")
    diagnoses = case.get("diagnoses") or []
    grade = np.nan
    stage = np.nan
    for diagnosis in diagnoses:
        if pd.isna(grade):
            grade = parse_grade(diagnosis.get("tumor_grade"))
        if pd.isna(stage):
            stage = parse_stage(diagnosis.get("ajcc_pathologic_stage"))
        if pd.notna(grade) and pd.notna(stage):
            break
    clinical_rows.append({
        "sample_id": case_id,
        "grade": grade,
        "stage": stage,
    })

tcga_clinical = pd.DataFrame(clinical_rows).drop_duplicates("sample_id")
tcga = tcga_gene.reset_index().merge(tcga_clinical, on="sample_id", how="left")
tcga.insert(0, "cohort", "TCGA-KIRC")

print("TCGA cases:", len(tcga))
print(tcga[["grade", "stage"]].notna().sum())


TCGA cases: 533
grade    525
stage    530
dtype: int64


In [16]:
cptac_rna = read_linkedomics(RAW["cptac_rna"])
cptac_protein = read_linkedomics(RAW["cptac_protein"])

for matrix_name, matrix in [("RNA", cptac_rna), ("protein", cptac_protein)]:
    missing = sorted(set(GENES) - set(matrix.index))
    assert not missing, f"CPTAC {matrix_name} missing {missing}"

cptac_gene = cptac_rna.loc[GENES].T
cptac_gene.index.name = "sample_id"
cptac_gene["mitoxy"] = equal4_score(cptac_gene)

cptac_clinical_raw = pd.read_excel(
    RAW["cptac_clinical_xlsx"],
    sheet_name="ccrcc_clinical_characteristics",
    header=0,
    dtype=str,
)
cptac_clinical_raw.columns = [str(column).strip() for column in cptac_clinical_raw.columns]

# Table S1 contains seven non-clear-cell RCC cases. Figure 1 uses only the
# 103 histologically confirmed clear-cell RCC tumors.
histology_column = column_containing(
    cptac_clinical_raw.columns,
    ["histologic_type", "histologic type"],
)
cptac_clinical_raw = cptac_clinical_raw.loc[
    cptac_clinical_raw[histology_column]
    .astype(str)
    .str.strip()
    .str.casefold()
    .eq("clear cell renal cell carcinoma")
].copy()
assert len(cptac_clinical_raw) == 103, (
    "Expected 103 clear-cell RCC cases in CPTAC Table S1; "
    f"found {len(cptac_clinical_raw)}"
)

# Identify the case-ID column by overlap with expression sample IDs.
sample_ids = set(cptac_gene.index.astype(str))
overlap_counts = {}
for column in cptac_clinical_raw.columns:
    values = set(cptac_clinical_raw[column].dropna().astype(str).str.strip())
    overlap_counts[column] = len(values & sample_ids)
id_column = max(overlap_counts, key=overlap_counts.get)
assert overlap_counts[id_column] >= 100

grade_column = column_containing(cptac_clinical_raw.columns, ["grade"])
stage_column = column_containing(cptac_clinical_raw.columns, ["stage"])

cptac_clinical = pd.DataFrame({
    "sample_id": cptac_clinical_raw[id_column].astype(str).str.strip(),
    "grade": cptac_clinical_raw[grade_column].map(parse_grade),
    "stage": cptac_clinical_raw[stage_column].map(parse_stage),
}).drop_duplicates("sample_id")

cptac = cptac_gene.reset_index().merge(cptac_clinical, on="sample_id", how="inner")
cptac.insert(0, "cohort", "CPTAC")

print("CPTAC cases:", len(cptac))
print(cptac[["grade", "stage"]].notna().sum())


CPTAC cases: 103
grade    103
stage    103
dtype: int64


In [17]:
OUTPUT_DIR = (
    PROJECT_ROOT / "data" / "processed" / "supplementary_figure_s2"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

mutation_index_path = source_path(
    "CPTAC ccRCC", "GDC masked somatic mutation file index"
)
mutation_bundle_path = source_path(
    "CPTAC ccRCC", "GDC masked somatic mutation bundle"
)


def fit_hc3(data, outcome, predictors, focal):
    complete = (
        data[[outcome, *predictors]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )
    X = sm.add_constant(
        complete[predictors], has_constant="add"
    ).astype(float)
    fit = sm.OLS(
        complete[outcome].astype(float), X
    ).fit(cov_type="HC3", use_t=False)
    ci = fit.conf_int().loc[focal]
    return {
        "n": len(complete),
        "effect": float(fit.params[focal]),
        "se": float(fit.bse[focal]),
        "ci_low": float(ci.iloc[0]),
        "ci_high": float(ci.iloc[1]),
        "p_value": float(fit.pvalues[focal]),
    }


PROGRAMS = {
    "WAVE regulatory complex": ["NCKAP1", "ABI1", "CYFIP1", "WASF2", "BRK1"],
    "ARP2/3 complex": [
        "ACTR2", "ACTR3", "ARPC1A", "ARPC1B", "ARPC2", "ARPC3",
        "ARPC4", "ARPC5", "ARPC5L",
    ],
    "Hypoxia/glycolysis": ["SLC2A1", "CA9", "LDHA", "PGK1", "ENO1"],
    "Caspase/apoptosis": ["CASP3", "CASP7", "CASP8", "CASP9"],
    "Immune": ["CD3D", "CD3E", "CD8A", "PRF1", "GZMB"],
    "Stromal": ["COL1A1", "COL1A2", "COL3A1", "DCN", "COL6A1"],
    "Endothelial": ["PECAM1", "VWF", "KDR", "ESM1", "EMCN"],
    "ccRCC epithelial": ["CA9", "EPCAM", "KRT8", "KRT18", "NDUFA4L2"],
}


## S2A — grade association after competing-program adjustment


In [18]:
x=tcga_expression.copy(); target=list(dict.fromkeys(GENES+[g for v in PROGRAMS.values() for g in v])); target=[g for g in target if g in x.index]
g=x.loc[target].T.apply(pd.to_numeric,errors="coerce"); g.index=g.index.astype(str).str.upper(); g=g.loc[g.index.str.match(r"^TCGA-.+-.+-01")]
g["sample_id"]=g.index.str[:12]; g=g.groupby("sample_id")[target].mean(); z=g.apply(zscore,axis=0)
comp=pd.DataFrame(index=z.index); comp["mitoxy_raw"]=z[GENES].mean(axis=1)
comp["mitoxy_sd"]=zscore(comp["mitoxy_raw"])
for name,genes in PROGRAMS.items():
    present=[q for q in genes if q in z]
    if len(present)<3: raise KeyError(f"Too few genes for {name}: {present}")
    comp[name]=z[present].mean(axis=1)
comp=comp.reset_index().merge(tcga_clinical,on="sample_id",how="left")
panel_a=pd.DataFrame([{"model":"Competing-program adjusted","programs":"; ".join(PROGRAMS),
                       "effect_unit":"per cohort SD",
                       **fit_hc3(comp,"grade",["mitoxy_sd",*PROGRAMS],"mitoxy_sd")}])
panel_a.to_csv(OUTPUT_DIR/"panel_a_competing_programs.csv",index=False); display(panel_a)



,model,programs,effect_unit,n,effect,se,ci_low,ci_high,p_value
0,Competing-program adjusted,WAVE regulatory complex; ARP2/3 complex; Hypox...,per cohort SD,525,0.092764,0.0451,0.00437,0.181157,0.039699


## S2B — PBRM1 adjustment and prespecified strata


In [19]:
def canonical_cptac_id(value):
    text = str(value).strip().upper()
    match = re.search(r"(C3[NL]-\d{5})", text)
    return match.group(1) if match else text


with mutation_index_path.open(encoding="utf-8") as handle:
    gdc_index_payload = json.load(handle)
gdc_hits = gdc_index_payload.get("data", {}).get("hits", [])
if not gdc_hits:
    raise RuntimeError("The GDC CPTAC-3 mutation index contains no files")

file_to_case = {}
for hit in gdc_hits:
    cases = hit.get("cases", [])
    if len(cases) != 1 or not cases[0].get("submitter_id"):
        raise RuntimeError(
            f"GDC file {hit.get('file_id')} lacks one case submitter ID"
        )
    file_to_case[str(hit["file_id"])] = canonical_cptac_id(
        cases[0]["submitter_id"]
    )

cp = cptac[["sample_id", "mitoxy", "grade"]].copy()
cp["canonical_id"] = cp["sample_id"].map(canonical_cptac_id)
expression_ids = set(cp["canonical_id"])
indexed_case_ids = set(file_to_case.values())
id_overlap = len(expression_ids & indexed_case_ids)
if id_overlap < 20:
    raise RuntimeError(
        "CPTAC GDC mutation identifiers did not harmonize with expression "
        f"IDs: overlap={id_overlap}"
    )

pbrm1_ids = set()
pbrm1_row_count = 0
maf_files_read = 0
with tarfile.open(mutation_bundle_path, mode="r:gz") as archive:
    for member in archive.getmembers():
        if not member.isfile() or not member.name.endswith(".maf.gz"):
            continue
        file_id = member.name.split("/", 1)[0]
        case_id = file_to_case.get(file_id)
        if case_id not in expression_ids:
            continue
        extracted = archive.extractfile(member)
        if extracted is None:
            raise RuntimeError(f"Could not extract {member.name}")
        with gzip.GzipFile(fileobj=extracted) as handle:
            mutation_rows = pd.read_csv(
                handle,
                sep="\t",
                comment="#",
                usecols=["Hugo_Symbol", "Variant_Classification"],
                low_memory=False,
            )
        maf_files_read += 1
        is_pbrm1 = (
            mutation_rows["Hugo_Symbol"]
            .astype(str)
            .str.upper()
            .eq("PBRM1")
        )
        pbrm1_row_count += int(is_pbrm1.sum())
        if is_pbrm1.any():
            pbrm1_ids.add(case_id)

pbrm1_ids &= expression_ids
cp["PBRM1_mut"] = cp["canonical_id"].isin(pbrm1_ids).astype(float)
n_mut = int(cp["PBRM1_mut"].sum())
n_wt = int(cp["PBRM1_mut"].eq(0).sum())
if n_mut == 0 or n_wt == 0:
    raise RuntimeError(
        "PBRM1 classification has no variation after harmonization: "
        f"mutated={n_mut}, wild_type={n_wt}"
    )

# Use the same cohort-wide SD unit in the adjusted model and both strata.
cp["mitoxy_sd"] = zscore(cp["mitoxy"])
rows = [{
    "analysis": "PBRM1-adjusted",
    "stratum": "All",
    "supported": True,
    "effect_unit": "per cohort SD",
    **fit_hc3(cp, "grade", ["mitoxy_sd", "PBRM1_mut"], "mitoxy_sd"),
}]
for value, label in [(0.0, "PBRM1 wild type"), (1.0, "PBRM1 mutated")]:
    sub = cp.loc[cp["PBRM1_mut"].eq(value)].dropna(
        subset=["mitoxy_sd", "grade"]
    )
    supported = len(sub) >= 20 and sub["grade"].nunique() >= 2
    if supported:
        rows.append({
            "analysis": "Within stratum",
            "stratum": label,
            "supported": True,
            "effect_unit": "per cohort SD",
            **fit_hc3(sub, "grade", ["mitoxy_sd"], "mitoxy_sd"),
        })
    else:
        rows.append({
            "analysis": "Within stratum",
            "stratum": label,
            "supported": False,
            "effect_unit": "per cohort SD",
            "n": len(sub),
            "effect": np.nan,
            "se": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "p_value": np.nan,
        })

panel_b = pd.DataFrame(rows)
panel_b.to_csv(OUTPUT_DIR / "panel_b_pbrm1.csv", index=False)

pbrm1_audit = pd.DataFrame([{
    "mutation_gene_column": "Hugo_Symbol",
    "mutation_id_column": "GDC files.cases.submitter_id",
    "mutation_expression_id_overlap": id_overlap,
    "indexed_gdc_files": len(file_to_case),
    "matched_maf_files_read": maf_files_read,
    "pbrm1_rows": pbrm1_row_count,
    "pbrm1_mutated_cases": n_mut,
    "pbrm1_wild_type_cases": n_wt,
    "mutated_grade_evaluable": int(
        cp.loc[cp["PBRM1_mut"].eq(1), "grade"].notna().sum()
    ),
    "wild_type_grade_evaluable": int(
        cp.loc[cp["PBRM1_mut"].eq(0), "grade"].notna().sum()
    ),
}])
pbrm1_audit.to_csv(
    OUTPUT_DIR / "pbrm1_identifier_audit.csv", index=False
)
display(pbrm1_audit)
display(panel_b)



,mutation_gene_column,mutation_id_column,mutation_expression_id_overlap,indexed_gdc_files,matched_maf_files_read,pbrm1_rows,pbrm1_mutated_cases,pbrm1_wild_type_cases,mutated_grade_evaluable,wild_type_grade_evaluable
0,Hugo_Symbol,GDC files.cases.submitter_id,103,353,195,80,44,59,44,59


,analysis,stratum,supported,effect_unit,n,effect,se,ci_low,ci_high,p_value
0,PBRM1-adjusted,All,True,per cohort SD,103,0.296022,0.068420,0.161922,0.430121,0.000015
1,Within stratum,PBRM1 wild type,True,per cohort SD,59,0.340008,0.077278,0.188546,0.491471,0.000011
2,Within stratum,PBRM1 mutated,True,per cohort SD,44,0.225147,0.119322,-0.008720,0.459013,0.059176


In [20]:
contracts = {
    'panel_a_competing_programs.csv': ['model', 'n', 'effect', 'ci_low', 'ci_high', 'p_value', 'effect_unit'],
    'panel_b_pbrm1.csv': ['analysis', 'stratum', 'supported', 'n', 'effect', 'ci_low', 'ci_high', 'p_value', 'effect_unit'],
    'pbrm1_identifier_audit.csv': ['mutation_gene_column', 'mutation_id_column', 'mutation_expression_id_overlap', 'pbrm1_mutated_cases', 'pbrm1_wild_type_cases'],
}
rows=[]
for filename,required in contracts.items():
    path=OUTPUT_DIR/filename
    table=pd.read_csv(path)
    missing=sorted(set(required)-set(table.columns))
    if missing: raise KeyError(f"{filename} missing {missing}")
    rows.append({"file":filename,"rows":len(table),"size_bytes":path.stat().st_size,"sha256":sha256_file(path)})
processed_manifest=pd.DataFrame(rows)
processed_manifest.to_csv(OUTPUT_DIR/"processed_manifest.csv",index=False)
display(processed_manifest)
print("Processed-file handoff: PASS")


,file,rows,size_bytes,sha256
0,panel_a_competing_programs.csv,1,337,0de6a8126d467897b9d43b4ef0e6fdf0d191fe9723abc1...
1,panel_b_pbrm1.csv,3,523,7fcab3e738574a19d4859c87940155c7c17905b9529762...
2,pbrm1_identifier_audit.csv,1,283,1dba3889be3b0f319d5308e7b66231010ef46c394a7377...


Processed-file handoff: PASS
